**Import des bases de données**

In [ ]:
import pandas as pd
import numpy as np 
import sqlite3
import requests
from bs4 import BeautifulSoup

In [ ]:
# visualisation des tables
conn = sqlite3.connect("../data/bookworld_reference.db")
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

In [ ]:
#chargement des tables d'une base SQLite dans un un dictionnaire {nom_table: DataFrame}

def charger_tables(chemin_db):
    with sqlite3.connect(chemin_db) as conn:
        noms = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"]
        return {nom: pd.read_sql(f'SELECT * FROM "{nom}"', conn) for nom in noms}

tables = charger_tables("../data/bookworld_reference.db")

for nom, df in tables.items():
    print(f"{nom} : {df.shape[0]} lignes, {df.shape[1]} colonnes")

In [ ]:
df_channels = tables["channels"]
df_countries = tables["countries"]
df_category_rules = tables["category_rules"]

In [ ]:
df_raw = pd.read_csv("../data/sales_raw.csv")

In [ ]:
df_channels.info()

In [ ]:
df_countries.head(10)
df_countries.info()

In [ ]:
df_category_rules.info()

In [ ]:
df_raw.info()

**Analyse des DF et premières actions de nettoyage**

In [ ]:
verif = df_raw[["country_code"]].drop_duplicates().merge(
    df_countries[["country_code", "country_name", "is_active"]],
    on="country_code",
    how="outer",
    indicator=True
)
verif

Pas de vente au GB ni au Canada, des ventes au NL mais le pays n'est pas référencé. 
Le portugal a des ventes mais est référencé comme inactif, attention à ne pas utiliser ce filtre. 

**Webscraping**

In [ ]:
# récupération du contenu 
url = "https://books.toscrape.com/"
page = requests.get(url)
page.raise_for_status()

In [ ]:
# segmentation avec BS
soup = BeautifulSoup(page.content, "html.parser")
print(soup.prettify())

In [ ]:
# récupération de la liste des livres de la 1ère page 
livres = soup.find_all("article", class_="product_pod")
print(len(livres))   

In [ ]:
# récupération des titres et prix des 20 livres de la 1ère page (liste)
livres = soup.find_all("article", class_="product_pod")
liste =[]
for livre in livres : 
    titre = livre.find("h3").find("a")["title"]
    prix = livre.find("p", class_="price_color").get_text(strip=True)
    liste.append({"book_name": titre, "price_gbp": prix})

In [ ]:
# transformation de ma liste en dataframe
df_catalogue = pd.DataFrame(liste)
df_catalogue.shape     
df_catalogue.dtypes    

In [ ]:
# nettoyage de la colonne de prix (retrait du sigle et passage en float)
df_catalogue["price_gbp"] = df_catalogue["price_gbp"].str.replace("£", "", regex=False).astype(float)


In [ ]:
# mise de la date au bon format 
df_raw["order_date"] =  pd.to_datetime(df_raw["order_date"])

**Récupération du taux de change à date (25/11/2026) entre Livres et Euros**
Taux de change : Frankfurter (frankfurter.dev)

In [ ]:
import requests

In [ ]:
# Utilisation de l'API de conversion pour le taux d'une paire (intégration des 2 devises directement dans l'url sans avoir besoin de params)
response = requests.get("https://api.frankfurter.dev/v2/rate/gbp/eur")
response.raise_for_status()
print(response.json())

In [ ]:
# récupération du taux à date 
taux = response.json()["rate"]
rate_date = response.json()["date"]
print(f"1 GBP = {taux} EUR le {rate_date}")


In [ ]:
df_catalogue["price_eur"] = df_catalogue["price_gbp"] * taux
df_catalogue["rate"] = taux


**Merge des df raw et catalogue pour associer le prix aux commandes**

In [ ]:
df_merged = df_raw.merge(df_catalogue, how='left', on='book_name')

# vérification que toutes les lignes ont bien une valeur car le merge sur le book_name n'était pas le plus fiable
df_merged.isna().sum()

**Ajout de la colonne 'country_name' depuis le df countries**

In [ ]:
# merge spécifique, les autres colonnes ne seront pas utilisées pour la suite (calcul hors TVA, is_active n'est pas un indicateur fiable)
df_merged = df_merged.merge(df_countries[["country_code", "country_name"]], on="country_code", how="left")

In [ ]:
# Vérification : tous les pays ont il bien un nom?
print(df_merged['country_name'].isna().sum())
df_merged[df_merged["country_name"].isna()]


In [ ]:

# remplacement des NaN par le nom du pays associé 
corrections = {"NL" : "Netherlands"}
df_merged["country_name"] = df_merged["country_name"].fillna(df_merged["country_code"].map(corrections))

**Calculs des revenus bruts et nets**

In [ ]:
# calculs
df_merged["gross_revenue_gbp"] = df_merged['quantity'] * df_merged['price_gbp']
df_merged["gross_revenue_eur"] = df_merged['quantity'] * df_merged['price_eur']
df_merged["net_revenue_gbp"] = df_merged["gross_revenue_gbp"] * (1 - df_merged['discount_rate'])
df_merged["net_revenue_eur"] = df_merged["gross_revenue_eur"] * (1 - df_merged['discount_rate'])

In [ ]:
# Vérifications
df_merged[["gross_revenue_gbp", "net_revenue_gbp"]].isna().sum()   
(df_merged["net_revenue_gbp"] <= df_merged["gross_revenue_gbp"]).all() 

**Préparation de la création du DF final (sales_by_country)**

In [ ]:
# vérification que tous les order_id sont bien uniques pour compter le nombre de commandes 
df_merged.duplicated(subset=["order_id"]).sum()

In [ ]:
# agrégations, arrondis et tri 
sales_by_country = (
    df_merged
    .groupby(['country_code','country_name'])
    .agg(
    total_orders=('order_id', 'nunique'),
    total_quantity=('quantity', 'sum'),
    total_revenue_gbp=('net_revenue_gbp','sum'),
    total_revenue_eur=('net_revenue_eur','sum'),
    total_gross_revenue_gbp=('gross_revenue_gbp','sum'),
    total_gross_revenue_eur=('gross_revenue_eur','sum'))
    .reset_index()
    .round(2)
    .sort_values(by='total_revenue_eur', ascending=False, ignore_index=True)
)

In [ ]:
sales_by_country['total_orders'].sum()

In [ ]:
sales_by_country.shape

In [ ]:
sales_by_country.describe()

In [ ]:
sales_by_country.head(8)

**Création de la base finale à partir du schéma**

In [ ]:
# Création de la base finale à partir du schéma
conn_final = sqlite3.connect("../bookworld_final.db")
conn_final.execute("PRAGMA foreign_keys = ON")

with open("../schema_final.sql", encoding="utf-8") as f:
    conn_final.executescript(f.read())

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn_final)